# Clustering Analysis

Downstream analysis demonstrating clustering of extracted scientific data.
We cluster entity-attribute feature vectors from (a) ground truth, (b) full LLM
extraction, and (c) synthetic probe data, then measure how confidence-weighted
KMeans — using probe or NTP scores — aligns extracted clusters with ground truth.

In [ ]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

from analysis.loaders import (
    load_combined_judgements, load_extraction, load_ground_truth,
    load_trained_probe, load_activations,
    load_synthetic_responses, load_synthetic_activations,
)
from scholarlm.utils.unit_conversion import apply_unit_conversion
from experiments.run_extraction import load_dataset_config
import paths

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "text.usetex": False,
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "legend.title_fontsize": 9,
    "axes.linewidth": 0.6,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "lines.linewidth": 1.2, "lines.markersize": 4,
    "legend.frameon": False,
    "figure.dpi": 150, "savefig.dpi": 300,
    "savefig.format": "pdf", "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

FIGURES_DIR = Path("../figures/clustering/")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
DATASET          = 'pond'
EXTRACTION_MODEL = 'gemma-3-27b'
EXTRACTION_DATE  = '2026_05_05'
JUDGE_MODEL      = 'llama-3.1-8b'
JUDGE_DATE       = '2026_05_06'
SYN_JUDGE_DATE   = '2026_05_04'  # date for synthetic probe activations
SYN_SPLIT        = 'test'         # 'test' = held-out papers not in probe training

N_CLUSTERS = 6  # chosen from elbow curve below

## 1. Load Raw Data

In [ ]:
config  = load_dataset_config(DATASET)
gt_df   = load_ground_truth(config)
records = load_extraction(DATASET, EXTRACTION_MODEL, EXTRACTION_DATE)
ext_df  = pd.DataFrame(records)
syn_responses = load_synthetic_responses(DATASET, JUDGE_MODEL, SYN_JUDGE_DATE, split=SYN_SPLIT)
syn_df  = pd.DataFrame(syn_responses)

# Unit conversion
gt_df  = apply_unit_conversion(gt_df,  config.unit_conversion_table)
ext_df = apply_unit_conversion(ext_df, config.unit_conversion_table)
syn_df = apply_unit_conversion(syn_df, config.unit_conversion_table)

# GT needs a synthetic entity_id (ext_df and syn_df already carry one)
_GT_ENTITY_COLS = ['document_id', 'name', 'location', 'ecosystem']
gt_df['entity_id'] = gt_df.groupby(_GT_ENTITY_COLS, dropna=False).ngroup()

print(f"Loaded: gt={len(gt_df):,} rows  |  ext={len(ext_df):,} rows  |  syn={len(syn_df):,} rows")
print(f"Entities — GT: {gt_df['entity_id'].nunique():,},  "
      f"Ext: {ext_df['entity_id'].nunique():,},  "
      f"Syn: {syn_df['entity_id'].nunique():,}")
print(f"Attributes: {sorted(gt_df['attribute'].unique())}")

## 2. Filter Probe Training Documents

The probe was trained on synthetic measurements derived from a specific set of papers.
We remove those papers from all three datasets before clustering to prevent in-sample bias.

In [ ]:
pd_data     = load_trained_probe(DATASET, JUDGE_MODEL)
syn_doc_ids = set(pd_data['syn_document_ids'])
top_k       = pd_data['top_k_heads']

n_gt_before  = len(gt_df)
n_ext_before = len(ext_df)
n_syn_before = len(syn_df)

gt_df  = gt_df[ ~gt_df['document_id'].isin(syn_doc_ids)].reset_index(drop=True)
ext_df = ext_df[~ext_df['document_id'].isin(syn_doc_ids)].reset_index(drop=True)
syn_df = syn_df[~syn_df['document_id'].isin(syn_doc_ids)].reset_index(drop=True)

# Recompute entity_id for gt after row removal (ngroup values shift)
gt_df['entity_id'] = gt_df.groupby(_GT_ENTITY_COLS, dropna=False).ngroup()

print(f"Probe trained on {len(syn_doc_ids)} documents.")
print(f"Row counts after filtering:")
print(f"  GT : {n_gt_before:,} → {len(gt_df):,}  ({n_gt_before - len(gt_df):,} removed)")
print(f"  Ext: {n_ext_before:,} → {len(ext_df):,}  ({n_ext_before - len(ext_df):,} removed)")
print(f"  Syn: {n_syn_before:,} → {len(syn_df):,}  ({n_syn_before - len(syn_df):,} removed)")

## 3. Data Quality

Inspect missing rates per attribute to decide which columns and rows to keep.
Attributes above `COL_MISS_THRESHOLD` in GT are dropped; entity rows that are
entirely missing after column removal are also dropped.

In [ ]:
_attrs = sorted(gt_df['attribute'].unique())

def _miss_rates(df, attrs):
    piv = df.pivot_table(index='entity_id', columns='attribute',
                         values='converted_value', aggfunc='first')
    piv = piv.reindex(columns=attrs)
    return piv.isna().mean(), len(piv)

gt_miss,  n_gt_ent  = _miss_rates(gt_df,  _attrs)
ext_miss, n_ext_ent = _miss_rates(ext_df, _attrs)
syn_miss, n_syn_ent = _miss_rates(syn_df, _attrs)

miss_summary = pd.DataFrame({
    'GT missing':  gt_miss,
    'Ext missing': ext_miss,
    'Syn missing': syn_miss,
}).round(3)
print("Missing rate per attribute (fraction of entities):")
print(miss_summary.to_string())
print(f"\nEntity counts — GT: {n_gt_ent:,},  Ext: {n_ext_ent:,},  Syn: {n_syn_ent:,}")

In [ ]:
from itertools import combinations

gt_wide = gt_df.pivot_table(
    index='entity_id', columns='attribute', values='converted_value', aggfunc='first'
)

# Enumerate all 2- and 3-column subsets; count fully observed GT rows for each
enum_rows = []
for d in (2, 3):
    for cols in combinations(gt_wide.columns, d):
        n = int(gt_wide[list(cols)].dropna().shape[0])
        enum_rows.append({'attributes': list(cols), 'd': d, 'n_complete': n, 'n×d': n * d})

enum_df = pd.DataFrame(enum_rows)

print("── d = 2  (top 8 by n×d) ──")
top2 = enum_df[enum_df['d'] == 2].sort_values('n×d', ascending=False).head(8)
print(top2[['attributes', 'n_complete', 'n×d']].to_string(index=False))

print("\n── d = 3  (top 8 by n×d) ──")
top3 = enum_df[enum_df['d'] == 3].sort_values('n×d', ascending=False).head(8)
print(top3[['attributes', 'n_complete', 'n×d']].to_string(index=False))

# Select best overall (highest n×d across both groups)
best = enum_df.sort_values('n×d', ascending=False).iloc[0]
keep_attrs = best['attributes']
print(f"\n▶ Selected: {keep_attrs}")
print(f"  GT complete entities: {best['n_complete']:,}   n×d = {best['n×d']:,}")

## 4. Feature Matrices

Build entity × attribute wide-format matrices for GT, extraction, and synthetic data.

**Alignment note**: Judge probabilities (NTP and probe) are merged into `ext_df` and
`syn_df` by `measurement_id` *before* pivoting, so all three value columns (numeric
value, NTP probability, probe probability) use the same `aggfunc='first'` selection
per entity × attribute cell — guaranteeing exact row-level alignment.

In [ ]:
# ── Load judge data; compute probe probabilities for extraction ───────────────
judge_df = pd.DataFrame(load_combined_judgements(DATASET, EXTRACTION_MODEL, EXTRACTION_DATE))
judge_df = judge_df[~judge_df['document_id'].isin(syn_doc_ids)].reset_index(drop=True)

act      = load_activations(DATASET, EXTRACTION_MODEL, EXTRACTION_DATE, JUDGE_MODEL, JUDGE_DATE)
mids_ext = judge_df['measurement_id'].tolist()
X_feat_ext = np.concatenate([
    np.stack([np.array(act[str(mid)], dtype=np.float32)[l, h, :] for mid in mids_ext], axis=0)
    for l, h in top_k
], axis=1)
judge_df['p_probe'] = pd_data['probe'].predict_proba(X_feat_ext)[:, 1]

# Merge judge probs into ext_df so pivoting stays aligned
ext_df = ext_df.merge(
    judge_df[['measurement_id', f'judgement_p_true_{JUDGE_MODEL}', 'p_probe']],
    on='measurement_id', how='left',
)

# ── Compute probe probabilities for synthetic data ─────────────────────────────
syn_act  = load_synthetic_activations(DATASET, JUDGE_MODEL, SYN_JUDGE_DATE, split=SYN_SPLIT)
mids_syn = syn_df['measurement_id'].tolist()
X_feat_syn = np.concatenate([
    np.stack([np.array(syn_act[str(mid)], dtype=np.float32)[l, h, :] for mid in mids_syn], axis=0)
    for l, h in top_k
], axis=1)
syn_df['p_probe'] = pd_data['probe'].predict_proba(X_feat_syn)[:, 1]

# ── Pivot all datasets onto keep_attrs (set by enumeration cell above) ─────────
def _pivot(df, value_col):
    piv = df.pivot_table(index='entity_id', columns='attribute',
                         values=value_col, aggfunc='first')
    piv.columns.name = None
    return piv.reindex(columns=keep_attrs)

gt_val_piv    = _pivot(gt_df,  'converted_value')
ext_val_piv   = _pivot(ext_df, 'converted_value')
ext_ntp_piv   = _pivot(ext_df, f'judgement_p_true_{JUDGE_MODEL}')
ext_probe_piv = _pivot(ext_df, 'p_probe')
syn_val_piv   = _pivot(syn_df, 'converted_value')
syn_ntp_piv   = _pivot(syn_df, 'judgement_p_true')
syn_probe_piv = _pivot(syn_df, 'p_probe')

# ── GT: keep only fully observed rows (the max-complete submatrix) ─────────────
gt_val_piv = gt_val_piv.dropna()

# ── Ext / Syn: drop only entities with no values at all; impute the rest ───────
ext_val_piv = ext_val_piv[~ext_val_piv.isna().all(axis=1)]
syn_val_piv = syn_val_piv[~syn_val_piv.isna().all(axis=1)]
ext_ntp_piv   = ext_ntp_piv.reindex(ext_val_piv.index)
ext_probe_piv = ext_probe_piv.reindex(ext_val_piv.index)
syn_ntp_piv   = syn_ntp_piv.reindex(syn_val_piv.index)
syn_probe_piv = syn_probe_piv.reindex(syn_val_piv.index)

print(f"Entities — GT (complete): {len(gt_val_piv):,},  "
      f"Ext: {len(ext_val_piv):,},  Syn: {len(syn_val_piv):,}")

In [ ]:
# ── Imputer fit on complete GT; scaler fit on GT ──────────────────────────────
# GT is already fully observed — imputer learns GT distribution to fill ext/syn gaps
imputer = KNNImputer(n_neighbors=5, weights='distance')
scaler  = StandardScaler()

X_gt  = scaler.fit_transform(imputer.fit_transform(gt_val_piv.to_numpy()))
X_ext = scaler.transform(imputer.transform(ext_val_piv.to_numpy()))
X_syn = scaler.transform(imputer.transform(syn_val_piv.to_numpy()))

# ── Entity-level probabilities (product across observed attributes) ────────────
def _entity_probs(prob_piv):
    return prob_piv.prod(axis=1, min_count=1).fillna(0.0).to_numpy()

ext_ntp_probs   = _entity_probs(ext_ntp_piv)
ext_probe_probs = _entity_probs(ext_probe_piv)
syn_ntp_probs   = _entity_probs(syn_ntp_piv)
syn_probe_probs = _entity_probs(syn_probe_piv)

print(f"Feature matrix shapes — GT: {X_gt.shape},  Ext: {X_ext.shape},  Syn: {X_syn.shape}")
print(f"Ext prob ranges — NTP:   [{ext_ntp_probs.min():.3f}, {ext_ntp_probs.max():.3f}]  "
      f"Probe: [{ext_probe_probs.min():.3f}, {ext_probe_probs.max():.3f}]")
print(f"Syn prob ranges — NTP:   [{syn_ntp_probs.min():.3f}, {syn_ntp_probs.max():.3f}]  "
      f"Probe: [{syn_probe_probs.min():.3f}, {syn_probe_probs.max():.3f}]")

## 5. Clustering

Fit KMeans on the ground-truth entities to define reference cluster centroids, then
sweep the weight exponent γ and measure how well confidence-weighted KMeans on the
extracted / synthetic data recovers those centroids.

**Centroid matching distance**: mean distance under the optimal (Hungarian) assignment
between GT centroids and extracted centroids.

In [ ]:
def centroid_matching_distance(A, B, metric='euclidean'):
    """Mean optimal-assignment distance between two sets of centroids."""
    D = cdist(A, B, metric=metric)
    row_ind, col_ind = linear_sum_assignment(D, maximize=False)
    return D[row_ind, col_ind].mean()

In [ ]:
inertia = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(X_gt)
    inertia.append(km.inertia_)

fig, ax = plt.subplots(figsize=(3.0, 2.4))
ax.plot(list(k_range), inertia, 'o-', ms=3.5, lw=1.2, color='#555555')
ax.axvline(N_CLUSTERS, lw=1.0, ls='--', color='#BB5566', alpha=0.9,
           label=f'$k = {N_CLUSTERS}$')
ax.set_xlabel('Number of clusters $k$')
ax.set_ylabel('Inertia')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f'elbow_{DATASET}.pdf')
plt.show()

In [ ]:
kmeans_gt  = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
gt_labels  = kmeans_gt.fit_predict(X_gt)
gt_centers = kmeans_gt.cluster_centers_

pca     = PCA(n_components=2)
X_gt_2d = pca.fit_transform(X_gt)

_CLUSTER_COLORS = [
    '#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677', '#AA3377',
    '#BBBBBB', '#44AA99', '#EE8866', '#AAAADD',
][:N_CLUSTERS]

fig, ax = plt.subplots(figsize=(3.2, 2.8))
for k in range(N_CLUSTERS):
    mask = gt_labels == k
    ax.scatter(X_gt_2d[mask, 0], X_gt_2d[mask, 1],
               c=_CLUSTER_COLORS[k], s=8, alpha=0.65, lw=0,
               label=f'Cluster {k + 1}')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend(fontsize=7, markerscale=1.8, ncol=2, loc='best',
          handletextpad=0.3, columnspacing=0.8, borderpad=0.4)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f'gt_clusters_{DATASET}.pdf')
plt.show()

In [ ]:
gamma_vals = np.linspace(1.0, 10.0, 101)

_sweep_configs = {
    'ext_ntp':   (X_ext, ext_ntp_probs),
    'ext_probe': (X_ext, ext_probe_probs),
    'syn_ntp':   (X_syn, syn_ntp_probs),
    'syn_probe': (X_syn, syn_probe_probs),
}
distances = {k: np.empty(len(gamma_vals)) for k in _sweep_configs}

for i, gamma in enumerate(gamma_vals):
    for key, (X, probs) in _sweep_configs.items():
        km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
        km.fit(X, sample_weight=probs ** gamma)
        distances[key][i] = centroid_matching_distance(gt_centers, km.cluster_centers_)

In [ ]:
# Tol vibrant palette — colorblind-friendly, high contrast
_STYLE = {
    'ext_ntp':   dict(color='#0077BB', ls='--', lw=1.3, alpha=0.85,
                      label='Extraction · NTP'),
    'ext_probe': dict(color='#0077BB', ls='-',  lw=1.8,
                      label='Extraction · Probe'),
    'syn_ntp':   dict(color='#EE7733', ls='--', lw=1.3, alpha=0.85,
                      label='Synthetic · NTP'),
    'syn_probe': dict(color='#EE7733', ls='-',  lw=1.8,
                      label='Synthetic · Probe'),
}

fig, ax = plt.subplots(figsize=(3.5, 2.8))

for key, style in _STYLE.items():
    ax.plot(gamma_vals, distances[key], **style)

ax.set_xlabel('Weight exponent $\\gamma$')
ax.set_ylabel('Centroid matching distance')
ax.set_xlim(gamma_vals[0], gamma_vals[-1])
ax.legend(fontsize=7.5, ncol=1, loc='upper right',
          handlelength=2.2, handletextpad=0.4)
ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))

fig.tight_layout()
fig.savefig(FIGURES_DIR / f'gamma_distance_{DATASET}.pdf')
plt.show()